# Hospital Site Contribution Matrix


In [1]:
import os
import sys
from pathlib import Path

root_dir = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import pandas as pd
import numpy as np

config = {
    "e1_csv": str(root_dir / "results" / "e1.csv"),
    "e2_csv": str(root_dir / "results" / "e2.csv"),
    "output_csv": str(root_dir / "results" / "e3.csv"),
    "rounds": 50,
}

Generate results/e3.csv by tagging leave-one-out evaluations under experiment e3.

In [2]:
df_e2 = pd.read_csv(config["e2_csv"])
df_e3 = df_e2.copy()
df_e3["exp"] = "e3"
df_e3["method"] = "fedavg_without_site"

os.makedirs(os.path.dirname(config["output_csv"]), exist_ok=True)
df_e3.to_csv(config["output_csv"], index=False)
print(f"Saved {len(df_e3)} rows to {config['output_csv']}")

Saved 8000 rows to C:\Users\Soumadipta Konar\Desktop\fl-hospital-consortium\results\e3.csv


Calculate paired differences across seeds: Delta = Acc(full) - Acc(without site j). Positive values indicate site j contributed positively to site k.

In [3]:
df_e1 = pd.read_csv(config["e1_csv"])

# Filter final round 50
full = df_e1[(df_e1["method"] == "fedavg") & (df_e1["round"] == config["rounds"]) & (df_e1["site"] != "all")].copy()
full["site"] = full["site"].astype(int)

without_j = df_e3[(df_e3["round"] == config["rounds"]) & (df_e3["site"] != "all")].copy()
without_j["site"] = without_j["site"].astype(int)
without_j["heldout_site"] = without_j["heldout_site"].astype(int)

merged = pd.merge(
    full[["seed", "site", "metric_value"]].rename(columns={"metric_value": "full_acc"}),
    without_j[["seed", "heldout_site", "site", "metric_value"]].rename(columns={"metric_value": "without_j_acc"}),
    on=["seed", "site"]
)

merged["delta"] = merged["full_acc"] - merged["without_j_acc"]
matrix = merged.groupby(["heldout_site", "site"])["delta"].agg(["mean", "std"]).reset_index()

site_names = {0: "cleveland (0)", 1: "hungarian (1)", 2: "switzerland (2)", 3: "va (3)"}
matrix["removed_site"] = matrix["heldout_site"].map(site_names)
matrix["evaluated_site"] = matrix["site"].map(site_names)
matrix["formatted"] = matrix.apply(lambda r: f"{r['mean']*100:+.2f}% (+/- {r['std']*100:.2f}%)", axis=1)

pivot_table = matrix.pivot(index="removed_site", columns="evaluated_site", values="formatted")
print("Site-by-Site Contribution Matrix (mean +/- std Delta Accuracy over 10 seeds):")
print("Positive (+%) means removing the row site hurt the column site (positive contribution).")
print(pivot_table.to_string())

Site-by-Site Contribution Matrix (mean +/- std Delta Accuracy over 10 seeds):
Positive (+%) means removing the row site hurt the column site (positive contribution).
evaluated_site        cleveland (0)       hungarian (1)     switzerland (2)              va (3)
removed_site                                                                                   
cleveland (0)    +2.98% (+/- 0.96%)  +1.24% (+/- 2.56%)  +0.62% (+/- 6.22%)  +5.56% (+/- 1.89%)
hungarian (1)    +0.58% (+/- 1.58%)  +2.47% (+/- 2.11%)  -5.00% (+/- 4.93%)  -8.89% (+/- 2.34%)
switzerland (2)  +0.19% (+/- 0.76%)  -0.45% (+/- 0.95%)  +0.62% (+/- 1.98%)  +0.67% (+/- 1.50%)
va (3)           +0.67% (+/- 0.91%)  +0.34% (+/- 1.99%)  +3.12% (+/- 4.42%)  +3.11% (+/- 3.00%)
